In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 17:35:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# This is what we want to implement now, but with RDDs
 ```
df_green_revenue = spark.sql("""
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    count(1) as number_records   
FROM
    green
where lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")
```

In [2]:
# read data 

df_green = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet("../../data/raw/green")
)  # read recursively 

In [3]:
# In the dataframe there is a field called rdd which is the internally underlying rdd of this df 

In [4]:
df_green.rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 12, 18, 15, 52, 30), lpep_dropoff_datetime=datetime.datetime(2019, 12, 18, 15, 54, 39), store_and_fwd_flag='N', RatecodeID=1.0, PULocationID=264, DOLocationID=264, passenger_count=5.0, trip_distance=0.0, fare_amount=3.5, extra=0.5, mta_tax=0.5, tip_amount=0.01, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=4.81, payment_type=1.0, trip_type=1.0, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 45, 58), lpep_dropoff_datetime=datetime.datetime(2020, 1, 1, 0, 56, 39), store_and_fwd_flag='N', RatecodeID=5.0, PULocationID=66, DOLocationID=65, passenger_count=2.0, trip_distance=1.28, fare_amount=20.0, extra=0.0, mta_tax=0.0, tip_amount=4.06, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=24.36, payment_type=1.0, trip_type=2.0, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 41, 38

In [5]:
df_green.take(5)  # also returns a list of rows,

# df is built on top of RDD of rows 
# Rows is a specific object used for building dataframes 
# we can do anything with RDDs
# we will look at couple operations next 

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2019, 12, 18, 15, 52, 30), lpep_dropoff_datetime=datetime.datetime(2019, 12, 18, 15, 54, 39), store_and_fwd_flag='N', RatecodeID=1.0, PULocationID=264, DOLocationID=264, passenger_count=5.0, trip_distance=0.0, fare_amount=3.5, extra=0.5, mta_tax=0.5, tip_amount=0.01, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=4.81, payment_type=1.0, trip_type=1.0, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 45, 58), lpep_dropoff_datetime=datetime.datetime(2020, 1, 1, 0, 56, 39), store_and_fwd_flag='N', RatecodeID=5.0, PULocationID=66, DOLocationID=65, passenger_count=2.0, trip_distance=1.28, fare_amount=20.0, extra=0.0, mta_tax=0.0, tip_amount=4.06, tolls_amount=0.0, ehail_fee=None, improvement_surcharge=0.3, total_amount=24.36, payment_type=1.0, trip_type=2.0, congestion_surcharge=0.0),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 41, 38

In [6]:
# keep only columns that are needed 
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [7]:
from datetime import datetime

start = datetime(year=2020, month=1, day=1)

In [8]:
# keep only records that are in 2020 or later 

rdd \
    .filter(lambda row: row.lpep_pickup_datetime >= start).take(1)  # this is how we implement the where condition 

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 45, 58), PULocationID=66, total_amount=24.36)]

In [9]:
# buts its bettter to put this into a function see below: 

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start



rdd \
    .filter(filter_outliers).take(1)  # keep only records that are in 2020 or later - this is how we implement the where condition


[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 1, 0, 45, 58), PULocationID=66, total_amount=24.36)]

In [10]:
# date_trunc('hour', lpep_pickup_datetime) AS hour,

# rows= rdd.take(10)
# row = rows[0]

def prepare_for_grouping(row): 
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)  # date_trunc
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)
    return (key, value)


rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .take(5)

[((datetime.datetime(2020, 1, 1, 0, 0), 66), (24.36, 1)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 181), (15.34, 1)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 129), (25.05, 1)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 210), (11.3, 1)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 35), (14.8, 1))]

In [11]:
# now we have to REDUCE: combining them into one 


def calculate_revenue(left_value, right_value):  # lerft_value, right_value comes from the output of map (key, VALUE)
    # VALUE is a tuple itself 
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .take(5)

[((datetime.datetime(2020, 1, 1, 0, 0), 256), (296.62000000000006, 13)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 24), (87.6, 3)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 244), (183.57999999999998, 12)),
 ((datetime.datetime(2020, 1, 1, 1, 0), 244), (255.85000000000002, 20)),
 ((datetime.datetime(2020, 1, 1, 0, 0), 146), (99.37, 6))]

In [14]:
# now we want to unnest this and turn this back into a df (because its not very pretty)

# this returns one tuple with all the elements 
def unwrap(row):
    return (row[0][0], row[0][1], row[1][0], row[1][1])


rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show()

+-------------------+---+------------------+---+
|                 _1| _2|                _3| _4|
+-------------------+---+------------------+---+
|2020-01-01 00:00:00|256|296.62000000000006| 13|
|2020-01-01 00:00:00| 24|              87.6|  3|
|2020-01-01 00:00:00|244|183.57999999999998| 12|
|2020-01-01 01:00:00|244|255.85000000000002| 20|
|2020-01-01 00:00:00|146|             99.37|  6|
|2020-01-01 00:00:00|198|            195.11|  5|
|2020-01-01 00:00:00|254| 73.82000000000001|  3|
|2020-01-01 01:00:00|264|             71.96|  4|
|2020-01-01 00:00:00|106|             10.56|  1|
|2020-01-01 01:00:00|188|181.03000000000003|  7|
|2020-01-01 01:00:00|146|             148.0| 11|
|2020-01-01 01:00:00| 70|              18.6|  2|
|2020-01-01 01:00:00| 40|177.16000000000003|  6|
|2020-01-01 01:00:00|152|             42.14|  4|
|2020-01-01 00:00:00|208|             80.24|  3|
|2020-01-01 01:00:00| 14|             68.83|  3|
|2020-01-01 01:00:00|208|             12.24|  1|
|2020-01-01 01:00:00

Traceback (most recent call last):
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [13]:
# i am encountering the issue it could be a perf problem running spark in notebook or something but i dont understand this 

In [17]:
# also we have no column names and also the schema is lost
# to make it easier for it , we can use a NAMED TUPLE 

from collections import namedtuple

RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

def unwrap(row):
    return RevenueRow(
        hour = row[0][0],
        zone = row[0][1],
        revenue = row[1][0],
        count = row[1][1]        
    )


rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF() \
    .show(5)

[Stage 18:====================================================>   (13 + 1) / 14]

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-01 00:00:00| 256|296.62000000000006|   13|
|2020-01-01 00:00:00|  24|              87.6|    3|
|2020-01-01 00:00:00| 244|183.57999999999998|   12|
|2020-01-01 01:00:00| 244|255.85000000000002|   20|
|2020-01-01 00:00:00| 146|             99.37|    6|
+-------------------+----+------------------+-----+
only showing top 5 rows


Traceback (most recent call last):                                              
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [18]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF()

In [ ]:
# now we want to provide schema

In [19]:
df_result.schema

StructType([StructField('hour', TimestampType(), True), StructField('zone', LongType(), True), StructField('revenue', DoubleType(), True), StructField('count', LongType(), True)])

In [22]:
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)]
)

df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

In [23]:
df_result.show()

[Stage 24:====================================================>   (13 + 1) / 14]

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-01 00:00:00| 256|296.62000000000006|   13|
|2020-01-01 00:00:00|  24|              87.6|    3|
|2020-01-01 00:00:00| 244|183.57999999999998|   12|
|2020-01-01 01:00:00| 244|255.85000000000002|   20|
|2020-01-01 00:00:00| 146|             99.37|    6|
|2020-01-01 00:00:00| 198|            195.11|    5|
|2020-01-01 00:00:00| 254| 73.82000000000001|    3|
|2020-01-01 01:00:00| 264|             71.96|    4|
|2020-01-01 00:00:00| 106|             10.56|    1|
|2020-01-01 01:00:00| 188|181.03000000000003|    7|
|2020-01-01 01:00:00| 146|             148.0|   11|
|2020-01-01 01:00:00|  70|              18.6|    2|
|2020-01-01 01:00:00|  40|177.16000000000003|    6|
|2020-01-01 01:00:00| 152|             42.14|    4|
|2020-01-01 00:00:00| 208|             80.24|    3|
|2020-01-01 01:00:00|  14|             68.83|    3|
|2020-01-01 

Traceback (most recent call last):                                              
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/.venv/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [24]:
# reason that i save it is that i want to look at the execution graph
df_result.write.parquet('tmp/green-revenue')

# it also has 2 stages like in the sql example 
# 1 x reduceByKey = reshuffling (because we have a bunch of partitions and for each partition we apply map function to turn the recurds into (H,Z)
# 

26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/10 17:50:46 WARN MemoryManager: Total allocation exceeds 95.

In [ ]:
# this is how things were done many years ago, now we have df and sql we dont need to write code like that
# but this is how low level operations work 